# M00 · Entorno, estructura y rutas — SOLUCIÓN

**Antes de ejecutar nada:** confirma arriba a la derecha que el kernel seleccionado es
`.venv`. Si dice "Python 3.x.x" a secas o apunta a otra ruta, cámbialo ahora — el
TODO 1 te dirá si acertaste.

---

> **Este es el notebook resuelto.** Ábrelo después de intentar `00_practica.ipynb`,
> no antes. Lo valioso aquí no es el código: son las notas que explican **por qué**
> cada decisión es la correcta.

Teoría de apoyo: `00_teoria.md` en esta misma carpeta.

---

## TODO 1 · Verificar el entorno

Lo primero es confirmar que el notebook está usando el Python correcto y que las
librerías instaladas cumplen los mínimos del plan.

In [ ]:
# === TODO 1 — SOLUCION ===

import sys
import sklearn

ejecutable = sys.executable
version_sklearn = sklearn.__version__


# --- verificacion (no modificar) ---
assert isinstance(ejecutable, str), "ejecutable debe ser un string"
assert isinstance(version_sklearn, str), "version_sklearn debe ser un string"

mayor, menor = (int(p) for p in version_sklearn.split(".")[:2])
assert (mayor, menor) >= (1, 5), (
    f"Este curso necesita scikit-learn >= 1.5 y tienes {version_sklearn}. "
    "Ejecuta: pip install -U scikit-learn"
)

print("interprete       :", ejecutable)
print("scikit-learn     :", version_sklearn)

if ".venv" in ejecutable or "venv" in ejecutable:
    print("\nOK: el kernel apunta al entorno virtual del proyecto.")
else:
    print(
        "\nATENCION: el interprete no parece ser el .venv del proyecto."
        "\nCambia el kernel arriba a la derecha antes de seguir."
    )

**Por qué `sys.executable` y no `pip list`.**
`pip list` te dice qué hay instalado en el Python que resuelve el comando `pip` en la
terminal. El notebook puede estar usando otro. `sys.executable` es la única fuente de
verdad sobre qué intérprete ejecuta *estas celdas*.

---

## TODO 2 · Resolver la raíz del proyecto

Ninguna ruta absoluta escrita a mano en un notebook sobrevive a un cambio de carpeta.
La alternativa es calcular la raíz en tiempo de ejecución.

In [ ]:
# === TODO 2 — SOLUCION ===

from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "requirements.txt").exists():
    if RAIZ == RAIZ.parent:          # llegamos a la raiz del disco: no esta
        raise FileNotFoundError("No se encontro requirements.txt en ninguna carpeta padre")
    RAIZ = RAIZ.parent

RUTA_DATA = RAIZ / "data"


# --- verificacion (no modificar) ---
assert isinstance(RAIZ, Path), "RAIZ debe ser un objeto Path"
assert (RAIZ / "requirements.txt").exists(), f"No hay requirements.txt en {RAIZ}"
assert (RAIZ / "src" / "utils.py").exists(), f"No se ve src/utils.py bajo {RAIZ}"
assert RUTA_DATA == RAIZ / "data", "RUTA_DATA debe apuntar a la carpeta data/"

print("raiz del proyecto:", RAIZ)
print("carpeta de datos :", RUTA_DATA)
print("existe data/     :", RUTA_DATA.exists())

**La guarda `if RAIZ == RAIZ.parent`.**
Sin ella, si ejecutas el notebook fuera del proyecto el bucle sube hasta `C:\` y se
queda girando para siempre, porque el padre de la raíz del disco es la raíz del disco.
Una versión equivalente sin bucle explícito:

```python
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "requirements.txt").exists())
```

---

## TODO 3 · Importar tu propio código

`src/utils.py` trae los helpers que vas a reutilizar en los dieciocho módulos. Un
notebook no lo encuentra por sí solo: hay que decirle a Python dónde buscar.

In [ ]:
# === TODO 3 — SOLUCION ===

import sys

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.utils import raiz_proyecto, estilo_graficos, resumen_bunch, dispersion_2d

estilo_graficos()


# --- verificacion (no modificar) ---
assert str(RAIZ) in sys.path, "RAIZ todavia no esta en sys.path"
assert sys.path.count(str(RAIZ)) == 1, "RAIZ esta duplicada en sys.path"

for f in (raiz_proyecto, estilo_graficos, resumen_bunch, dispersion_2d):
    assert callable(f), f"{f} no es invocable"

assert raiz_proyecto() == RAIZ, "raiz_proyecto() no coincide con tu RAIZ del TODO 2"

import matplotlib.pyplot as plt
assert plt.rcParams["axes.grid"] is True, "Parece que no llamaste a estilo_graficos()"

print("OK: src.utils importado y estilo aplicado.")
print("raiz_proyecto() ->", raiz_proyecto())

**Por qué `src` es importable.**
Porque tiene un `__init__.py`, que le dice a Python que esa carpeta es un paquete.
Sin ese archivo, `from src.utils import ...` falla aunque la ruta esté en `sys.path`.

**Si cambias `utils.py` mientras el notebook corre**, el cambio no se ve: Python cachea
los módulos importados. Añade al principio del notebook:

```python
%load_ext autoreload
%autoreload 2
```

---

## TODO 4 · Cargar iris y explorar el `Bunch`

`load_iris()` no devuelve una tupla ni un DataFrame: devuelve un `Bunch`, un diccionario
cuyas claves también funcionan como atributos. Es el formato de todos los datasets
integrados, así que conviene conocerlo bien desde el primer día.

In [ ]:
# === TODO 4 — SOLUCION ===

from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target
nombres_clases = iris.target_names
nombres_atributos = iris.feature_names


# --- verificacion (no modificar) ---
import numpy as np

assert X.shape == (150, 4), f"X deberia ser (150, 4) y es {X.shape}"
assert y.shape == (150,), f"y deberia ser (150,) y es {y.shape}"
assert len(nombres_clases) == 3, "Deberian ser 3 especies"
assert len(nombres_atributos) == 4, "Deberian ser 4 atributos"
assert set(np.unique(y)) == {0, 1, 2}, "y deberia contener las clases 0, 1 y 2"

resumen_bunch(iris, "iris")
print()
print("primera muestra :", X[0], "-> clase", y[0], f"({nombres_clases[y[0]]})")

**Tres formas de pedir el mismo dataset.**

```python
iris = load_iris()                       # Bunch completo: data, target, DESCR, ...
X, y = load_iris(return_X_y=True)        # solo los dos arrays
df = load_iris(as_frame=True).frame      # DataFrame de pandas, atributos + target
```

La tercera es cómoda para explorar, pero ojo: `frame` incluye la columna `target`, así
que si se la pasas entera a un modelo le estás dando la respuesta. En M05 volvemos
sobre esto con `ColumnTransformer`.

**`print(iris.DESCR)`** imprime la ficha del dataset: origen, número de instancias, qué
significa cada atributo. Vale la pena leerla una vez.

---

## TODO 5 · Tu primer gráfico

El objetivo no es el gráfico en sí, sino la observación que produce — la misma que el
libro usa para justificar todo el capítulo 1.

In [ ]:
# === TODO 5 — SOLUCION ===

eje = dispersion_2d(
    X,
    y,
    col_x=0,                      # longitud del sepalo
    col_y=1,                      # ancho del sepalo
    nombres_atributos=nombres_atributos,
    nombres_clases=nombres_clases,
    titulo="Iris: sepalo, longitud vs ancho",
)


# --- verificacion (no modificar) ---
import matplotlib.pyplot as plt

assert isinstance(eje, plt.Axes), "eje deberia ser un Axes de matplotlib"
assert len(eje.collections) == 3, (
    f"Se esperaban 3 grupos de puntos (uno por especie) y hay {len(eje.collections)}"
)
assert eje.get_xlabel(), "El eje X no tiene etiqueta: pasa nombres_atributos"
assert eje.get_title(), "Ponle un titulo al grafico"

print("OK. Mira el grafico: que especie se separa limpiamente de las otras dos?")

**Lo que se ve:** *setosa* ocupa una esquina propia y no toca a nadie. *versicolor* y
*virginica* se mezclan.

**Lo que eso predice:** con solo estos dos atributos, cualquier clasificador acertará
casi el 100 % en setosa y se equivocará bastante entre las otras dos. En M02 vamos a
comprobarlo con números, y luego veremos cuánto mejora al usar los cuatro atributos en
lugar de dos.

El libro hace este mismo gráfico con un bucle `for i in xrange(len(colors))` y una lista
de colores a mano. `dispersion_2d` hace lo mismo detectando las clases sola, pero ábrela
en `src/utils.py` y verás que por dentro es exactamente el bucle del libro.

---

## TODO 6 · Traducir un fragmento del libro

Este es el ejercicio que vas a repetir, con variantes, durante todo el curso.

Abajo está la celda 8 del notebook
`referencia/scikit-learn-book/Chapter 1  - A Gentle Introduction to Machine Learning.ipynb`,
copiada tal cual. Está escrita para Python 2 y scikit-learn 0.15, y tiene **dos** cosas
que hoy fallan:

```python
from sklearn.cross_validation import train_test_split
from sklearn.preprocessing import StandardScaler

# Get dataset with only the first two attributes
X, y = X_iris[:,:2], y_iris
# Split the dataset into a trainig and a testing set
# Test set will be the 25% taken randomly
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=33)
print X_train.shape, y_train.shape
# Standarize the features
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)

X_test = scaler.transform(X_test)
```

Antes de escribir: identifica las dos líneas que fallan y por qué.
Si dudas, la respuesta está en `docs/deprecaciones.md`, secciones 1 y 2.

In [ ]:
# === TODO 6 — SOLUCION ===

from sklearn.model_selection import train_test_split     # ya no: sklearn.cross_validation
from sklearn.preprocessing import StandardScaler

X_dos = X[:, :2]

X_train, X_test, y_train, y_test = train_test_split(
    X_dos, y, test_size=0.25, random_state=33
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform sobre TRAIN
X_test_s = scaler.transform(X_test)         # solo transform sobre TEST


# --- verificacion (no modificar) ---
import numpy as np

assert X_train.shape == (112, 2), f"X_train deberia ser (112, 2) y es {X_train.shape}"
assert X_test.shape == (38, 2), f"X_test deberia ser (38, 2) y es {X_test.shape}"
assert y_train.shape == (112,) and y_test.shape == (38,), "Formas de y incorrectas"

assert np.allclose(X_train_s.mean(axis=0), 0, atol=1e-9), (
    "La media de X_train_s deberia ser ~0 en cada columna. "
    "Ajustaste el scaler sobre el conjunto correcto?"
)
assert np.allclose(X_train_s.std(axis=0), 1, atol=1e-9), (
    "La desviacion de X_train_s deberia ser ~1 en cada columna."
)
assert not np.allclose(X_test_s.mean(axis=0), 0, atol=1e-6), (
    "La media de X_test_s ha salido exactamente 0: eso significa que ajustaste "
    "el scaler tambien sobre el test. Eso es data leakage."
)

print("formas   :", X_train_s.shape, X_test_s.shape)
print("train -> media", X_train_s.mean(axis=0).round(12), "desv", X_train_s.std(axis=0).round(12))
print("test  -> media", X_test_s.mean(axis=0).round(4), "desv", X_test_s.std(axis=0).round(4))
print("\nLa media del test NO es 0, y eso esta bien: se escalo con la media del train.")

**Las dos correcciones mecánicas:**

| Línea del libro | Por qué falla | Reemplazo |
| --- | --- | --- |
| `from sklearn.cross_validation import train_test_split` | El módulo se eliminó en 0.20 | `from sklearn.model_selection import train_test_split` |
| `print X_train.shape, y_train.shape` | Sintaxis de Python 2 | `print(X_train.shape, y_train.shape)` |

**Lo que el libro ya hacía bien, y es lo que importa.**
Fíjate en que el libro escribe `StandardScaler().fit(X_train)` y luego aplica
`.transform()` a los dos conjuntos por separado. Esa separación es la que evita el
*data leakage*, y es el motivo de que el último `assert` de la celda compruebe que la
media del test **no** sea cero.

Si hubieras escrito `scaler.fit_transform(X_test)`, ese assert habría saltado: la media
del test saldría 0 y tendrías una métrica optimista que no se sostiene con datos nuevos.

**Por qué 112 y 38.** 150 × 0.25 = 37.5, y scikit-learn redondea hacia arriba el test:
38 para test, 112 para entrenamiento.

**Por qué `random_state=33`.** Es el valor que usa el libro. Fijar la semilla hace el
reparto reproducible: sin ella obtendrías una división distinta en cada ejecución y no
podrías comparar resultados entre celdas. En M04 veremos por qué un solo reparto, por
muy reproducible que sea, no basta para estimar el rendimiento real.

**Lo que en M05 hará esto innecesario.** Todo este baile de `fit_transform` / `transform`
se resuelve solo dentro de un `Pipeline`, que aplica el escalado correctamente en cada
pliegue de la validación cruzada sin que tengas que pensarlo.

---

## Cierre del módulo

Si las seis celdas corrieron sin error, tienes:

- El kernel apuntando al entorno correcto y scikit-learn ≥ 1.5.
- Un patrón de rutas que no se rompe al mover el proyecto.
- `src/utils.py` importable desde cualquier notebook del curso.
- El formato `Bunch` de los datasets entendido.
- Tu primera traducción de código del libro a la API moderna.

### Antes de pasar a M01

1. Marca `M00` en el checklist del `README.md` de la raíz.
2. Anota en la bitácora qué te costó más.
3. Resuelve `00_ejercicios.md` — son cortos y consolidan lo de arriba.

**M01 · El contrato de la API de scikit-learn** es el módulo más importante del plan:
una vez que entiendas que toda la librería repite el mismo patrón `fit` / `transform` /
`predict`, podrás usar clases que nunca has visto sin leer la documentación entera.